In [1]:
import networkx as nx
import pandas as pd
import random
import numpy as np
import math
from cup_gather_data import hcgcr_data
import json
from tqdm import tqdm
from pathlib import Path
from unpack_graphs import unpack_graphs
from cup_gather_data import Iteration
from cup_main_gather_queue_data import updateCR
from cup_functions import update_queue_add_all_neighbors, \
    update_queue_remove_asymmetric, update_queue_remove_unchanged_orbits

base_path = Path(r'C:\Users\korol\OneDrive\Pulpit\Master\CUP - remove asymmetric, remove orbits, counting changes\result_graphs')
test_graphs = unpack_graphs()


In [3]:
def pair_generator(n): 
    used_pairs = set() 
    while True: 
        pair = random.sample(range(n), 2)
        pair = tuple(sorted(pair))
        if pair not in used_pairs: 
            used_pairs.add(pair) 
            yield pair 

def change_random_set_of_edges(G, s_len):
    G_new = G.copy()
    S = set()
    for _ in range(s_len):
        edge = next(pair_generator(len(G_new.nodes)))
        if G_new.has_edge(edge[0], edge[1]):
            G_new.remove_edge(edge[0], edge[1])
        else:
            G_new.add_edge(edge[0], edge[1])
        S.add(edge[0])
        S.add(edge[1])
    return G_new, S


def update_dict(G, S, iterations_up, q_lens):
    avg = (sum(q_lens) / len(q_lens)) if q_lens else 0.0
    data = {
        "n": G.number_of_nodes(),
        "m": G.number_of_edges(),
        "graph": json.dumps(list(G.edges())),
        "S": json.dumps(list(S)),
        "q_lens": json.dumps(q_lens),
        "q_len_average": avg,
        "coloring": json.dumps([
            {"color": it.coloring["color"].to_list(),
                "hash": it.coloring["hash"].to_list()}
            for it in iterations_up
        ]),
        "hash_dict": json.dumps([it.hash_dict for it in iterations_up], default=list)
    }
    return data


def random_graphs_up(s_p):
    out = {}
    for name, df in test_graphs.items():
        ups = []
        for row in tqdm(df.itertuples(index=False), desc="Generating for the file: "):
            G = row.graph
            s_len = max(1, math.ceil(G.number_of_edges() * s_p))
            G_up, S = change_random_set_of_edges(G, s_len)
            ups.append({"graph_up": G_up, "S": S})
        out[name] = ups
    return out


In [4]:
# only for the sets 10-150
print(test_graphs.keys())

graphs_001 = random_graphs_up(0.01)
graphs_005 = random_graphs_up(0.05)
graphs_020 = random_graphs_up(0.20)

dict_keys(['trees_10_150.csv', 'random_graphs_10_150_p_0.1.csv', 'random_graphs_10_150_p_0.3.csv', 'random_graphs_10_150_p_0.5.csv'])


Generating for the file: : 200it [00:00, 3156.39it/s]
Generating for the file: : 200it [00:00, 1639.07it/s]
Generating for the file: : 200it [00:00, 667.94it/s]
Generating for the file: : 200it [00:00, 439.49it/s]
Generating for the file: : 200it [00:00, 4297.31it/s]
Generating for the file: : 200it [00:00, 597.45it/s]
Generating for the file: : 200it [00:00, 609.03it/s]
Generating for the file: : 200it [00:00, 397.24it/s]
Generating for the file: : 200it [00:00, 3525.35it/s]
Generating for the file: : 200it [00:00, 1126.60it/s]
Generating for the file: : 200it [00:00, 443.75it/s]
Generating for the file: : 200it [00:00, 296.38it/s]


In [ ]:

def random_graphs_up(s_p):
    out = {}
    for name, df in test_graphs.items():
        ups = []
        for row in tqdm(df.itertuples(index=False), desc="Generating for the file: "):
            G = row.graph
            s_len = max(1, math.ceil(G.number_of_edges() * s_p))
            G_up, S = change_random_set_of_edges(G, s_len)
            ups.append({"graph_up": G_up, "S": S})
        out[name] = ups
    return out

graphs_001 = random_graphs_up(0.01)
graphs_005 = random_graphs_up(0.05)
graphs_020 = random_graphs_up(0.20)

Generating for the file: : 100it [00:00, 1694.47it/s]
Generating for the file: : 100it [00:00, 1126.41it/s]
Generating for the file: : 100it [00:00, 394.86it/s]
Generating for the file: : 100it [00:00, 265.95it/s]
Generating for the file: : 100it [00:00, 479.96it/s]
Generating for the file: : 100it [00:01, 91.15it/s]
Generating for the file: : 100it [01:07,  1.48it/s]
Generating for the file: : 100it [00:06, 16.26it/s]
Generating for the file: : 100it [00:00, 300.02it/s]
Generating for the file: : 100it [00:03, 26.55it/s]
Generating for the file: : 100it [00:10,  9.30it/s]
Generating for the file: : 100it [00:15,  6.31it/s]
Generating for the file: : 100it [00:00, 2095.66it/s]
Generating for the file: : 100it [00:00, 1029.68it/s]
Generating for the file: : 100it [00:00, 413.47it/s]
Generating for the file: : 100it [00:00, 256.76it/s]
Generating for the file: : 100it [00:00, 518.34it/s]
Generating for the file: : 100it [00:01, 87.78it/s]
Generating for the file: : 100it [00:03, 29.38it/

In [5]:
def update_dict(G, S, iterations_up, q_lens):
    avg = (sum(q_lens) / len(q_lens)) if q_lens else 0.0
    coloring_data = []
    for it in iterations_up:
        try:
            colors = it.coloring["color"].to_list()
        except Exception:
            colors = []

        try:
            hashes = it.coloring["hash"].to_list()
        except Exception:
            hashes = []
        coloring_data.append({"color": colors, "hash": hashes})
    
    data = {
        "n": G.number_of_nodes(),
        "m": G.number_of_edges(),
        "graph": json.dumps(list(G.edges())),
        "S": json.dumps(list(S)),
        "q_lens": json.dumps(q_lens),
        "q_len_average": avg,
        "coloring": json.dumps(coloring_data)
        #"hash_dict": json.dumps([it.hash_dict for it in iterations_up], default=list)
    }
    return data

In [6]:
def main_func(q_func, q_func_name, path: Path, up_graph_set):
    path.mkdir(parents=True, exist_ok=True)
    for name, df in test_graphs.items():
        ups = up_graph_set[name]  # list aligned with df rows
        result = []
        for i, row in enumerate(tqdm(df.itertuples(index=False), desc=f"{q_func_name} {name}")):
            G_up = ups[i]["graph_up"]
            S    = ups[i]["S"]
            iterations = row.iterations
            try:
                iterations_up, q_lens = updateCR(G_up, S, iterations, q_func)
                d = update_dict(G_up, S, iterations_up, q_lens)
            except Exception as e:
                print(f"Error in {q_func_name}, file '{name}', row {i}: {e}")
                # Optional: print traceback for debugging
                # import traceback; traceback.print_exc()
                d = {
                    "n": G_up.number_of_nodes(),
                    "m": G_up.number_of_edges(),
                    "graph": json.dumps(list(G_up.edges())),
                    "S": json.dumps([]),
                    "q_lens": json.dumps([]),
                    "q_len_average": 0,
                    "coloring": json.dumps([])
                }
            result.append(d)
        pd.DataFrame(result).to_csv(path / f"result_{q_func_name}_{name}", index=False)


In [7]:
# only for files 10-150
folnames = ["one_percent", "five_percent", "twenty_percent"]
paths = [base_path / fn for fn in folnames]
up_graphs_sets = [graphs_001, graphs_005, graphs_020]
q_func_names = ["cor", "cas", "cup"]
update_functions = [
    update_queue_remove_unchanged_orbits,
    update_queue_remove_asymmetric,
    update_queue_add_all_neighbors,
]
for path, up_graphs in zip(paths, up_graphs_sets):
    print(path)
    for qfn, q_func in zip(q_func_names, update_functions):
        print("We go with the update function:", qfn)
        try:
            main_func(q_func, qfn, path, up_graphs)
        except Exception as e:
            print("Error:", e)

C:\Users\korol\OneDrive\Pulpit\Master\CUP - remove asymmetric, remove orbits, counting changes\result_graphs\one_percent
We go with the update function: cor


cor trees_10_150.csv: 200it [04:16,  1.28s/it]
cor random_graphs_10_150_p_0.1.csv: 200it [00:34,  5.85it/s]
cor random_graphs_10_150_p_0.3.csv: 200it [00:28,  6.95it/s]
cor random_graphs_10_150_p_0.5.csv: 200it [00:30,  6.53it/s]


We go with the update function: cas


cas trees_10_150.csv: 200it [04:18,  1.29s/it]
cas random_graphs_10_150_p_0.1.csv: 200it [00:32,  6.08it/s]
cas random_graphs_10_150_p_0.3.csv: 200it [00:26,  7.54it/s]
cas random_graphs_10_150_p_0.5.csv: 200it [00:27,  7.38it/s]


We go with the update function: cup


...

cor trees_10_150.csv: 200it [06:23,  1.92s/it]
cor random_graphs_10_150_p_0.1.csv: 200it [00:48,  4.10it/s]
cor random_graphs_10_150_p_0.3.csv: 200it [00:35,  5.65it/s]
cor random_graphs_10_150_p_0.5.csv: 200it [00:33,  5.94it/s]


We go with the update function: cas


cas trees_10_150.csv: 200it [05:36,  1.68s/it]
cas random_graphs_10_150_p_0.1.csv: 200it [00:47,  4.17it/s]
cas random_graphs_10_150_p_0.3.csv: 200it [00:35,  5.59it/s]
cas random_graphs_10_150_p_0.5.csv: 200it [00:35,  5.71it/s]


We go with the update function: cup


cup trees_10_150.csv: 200it [06:45,  2.03s/it]
cup random_graphs_10_150_p_0.1.csv: 200it [03:12,  1.04it/s]
cup random_graphs_10_150_p_0.3.csv: 200it [03:16,  1.02it/s]
cup random_graphs_10_150_p_0.5.csv: 200it [03:10,  1.05it/s]


In [38]:
folnames = ["one_percent", "five_percent", "twenty_percent"]
paths = [base_path / fn for fn in folnames]
up_graphs_sets = [graphs_001, graphs_005, graphs_020]
q_func_names = ["cor", "cas", "cup"]
update_functions = [
    update_queue_remove_unchanged_orbits,
    update_queue_remove_asymmetric,
    update_queue_add_all_neighbors,
]
for path, up_graphs in zip(paths, up_graphs_sets):
    print(path)
    for qfn, q_func in zip(q_func_names, update_functions):
        print("We go with the update function:", qfn)
        try:
            main_func(q_func, qfn, path, up_graphs)
        except Exception as e:
            print("Error:", e)

C:\Users\korol\OneDrive\Pulpit\Master\CUP - remove asymmetric, remove orbits, counting changes\result_graphs\one_percent
We go with the update function: cor


cor trees_10_100.csv: 100it [02:36,  1.56s/it]
cor random_graphs_10_100_p_0.1.csv: 100it [00:36,  2.75it/s]
cor random_graphs_10_100_p_0.3.csv: 100it [00:22,  4.50it/s]
cor random_graphs_10_100_p_0.5.csv: 100it [00:22,  4.46it/s]
cor trees_150_300.csv: 100it [11:09,  6.69s/it]
cor random_graphs_150_300_p_0.1.csv: 100it [00:43,  2.31it/s]
cor random_graphs_150_300_p_0.3.csv: 100it [00:53,  1.88it/s]
cor random_graphs_150_300_p_0.5.csv: 100it [01:03,  1.58it/s]
cor trees_300_500.csv: 100it [23:59, 14.39s/it]
cor random_graphs_300_500_p_0.1.csv: 100it [01:28,  1.13it/s]
cor random_graphs_300_500_p_0.3.csv: 100it [02:13,  1.33s/it]
cor random_graphs_300_500_p_0.5.csv: 100it [02:32,  1.53s/it]


We go with the update function: cas


cas trees_10_100.csv: 100it [01:11,  1.40it/s]
cas random_graphs_10_100_p_0.1.csv: 100it [00:14,  6.91it/s]
cas random_graphs_10_100_p_0.3.csv: 100it [00:09, 10.94it/s]
cas random_graphs_10_100_p_0.5.csv: 100it [00:08, 11.16it/s]
cas trees_150_300.csv: 100it [10:46,  6.46s/it]
cas random_graphs_150_300_p_0.1.csv: 100it [00:38,  2.58it/s]
cas random_graphs_150_300_p_0.3.csv: 100it [00:54,  1.82it/s]
cas random_graphs_150_300_p_0.5.csv: 100it [01:04,  1.54it/s]
cas trees_300_500.csv: 100it [24:12, 14.52s/it]
cas random_graphs_300_500_p_0.1.csv: 100it [01:29,  1.12it/s]
cas random_graphs_300_500_p_0.3.csv: 100it [02:11,  1.31s/it]
cas random_graphs_300_500_p_0.5.csv: 100it [02:25,  1.46s/it]


We go with the update function: cup


...

cor trees_10_100.csv: 100it [01:18,  1.28it/s]
cor random_graphs_10_100_p_0.1.csv: 100it [00:25,  3.94it/s]
cor random_graphs_10_100_p_0.3.csv: 100it [00:10,  9.34it/s]
cor random_graphs_10_100_p_0.5.csv: 100it [00:10,  9.89it/s]
cor trees_150_300.csv: 100it [10:06,  6.06s/it]
cor random_graphs_150_300_p_0.1.csv: 100it [00:45,  2.19it/s]
cor random_graphs_150_300_p_0.3.csv: 100it [01:04,  1.54it/s]
cor random_graphs_150_300_p_0.5.csv: 100it [01:04,  1.54it/s]
cor trees_300_500.csv: 100it [22:29, 13.50s/it]
cor random_graphs_300_500_p_0.1.csv: 100it [01:45,  1.06s/it]
cor random_graphs_300_500_p_0.3.csv: 100it [02:41,  1.61s/it]
cor random_graphs_300_500_p_0.5.csv: 100it [02:33,  1.53s/it]


We go with the update function: cas


cas trees_10_100.csv: 100it [01:19,  1.26it/s]
cas random_graphs_10_100_p_0.1.csv: 100it [00:22,  4.53it/s]
cas random_graphs_10_100_p_0.3.csv: 100it [00:10,  9.38it/s]
cas random_graphs_10_100_p_0.5.csv: 100it [00:10,  9.89it/s]
cas trees_150_300.csv: 100it [10:16,  6.16s/it]
cas random_graphs_150_300_p_0.1.csv: 100it [00:47,  2.09it/s]
cas random_graphs_150_300_p_0.3.csv: 100it [01:03,  1.57it/s]
cas random_graphs_150_300_p_0.5.csv: 100it [01:07,  1.49it/s]
cas trees_300_500.csv: 100it [22:44, 13.64s/it]
cas random_graphs_300_500_p_0.1.csv: 100it [01:48,  1.08s/it]
cas random_graphs_300_500_p_0.3.csv: 100it [02:37,  1.57s/it]
cas random_graphs_300_500_p_0.5.csv: 100it [02:34,  1.54s/it]


We go with the update function: cup


cup trees_10_100.csv: 100it [01:41,  1.02s/it]
cup random_graphs_10_100_p_0.1.csv: 100it [00:57,  1.75it/s]
cup random_graphs_10_100_p_0.3.csv: 100it [00:57,  1.72it/s]
cup random_graphs_10_100_p_0.5.csv: 100it [00:56,  1.78it/s]
cup trees_150_300.csv: 100it [11:31,  6.91s/it]
cup random_graphs_150_300_p_0.1.csv: 100it [04:35,  2.76s/it]
cup random_graphs_150_300_p_0.3.csv: 100it [04:56,  2.96s/it]
cup random_graphs_150_300_p_0.5.csv: 100it [05:09,  3.09s/it]
cup trees_300_500.csv: 100it [24:38, 14.79s/it]
cup random_graphs_300_500_p_0.1.csv: 100it [09:23,  5.64s/it]
cup random_graphs_300_500_p_0.3.csv: 100it [10:35,  6.35s/it]
cup random_graphs_300_500_p_0.5.csv: 100it [10:23,  6.23s/it]


In [18]:
print(graphs_001["trees_10_100.csv"][0])

{'graph_up': <networkx.classes.graph.Graph object at 0x000001F47C583350>, 'S': {25, 34}}


In [ ]:
q_func = "update_queue_remove_asymmetric"
folname = "one_percent"
path = base_path / folname
up_graph_set = graphs_001

In [ ]:
q_func = "update_queue_add_all_neighbors"
folname = "one_percent"
path = base_path / folname
up_graph_set = graphs_001

In [ ]:
q_func = "update_queue_remove_unchanged_orbits"
folname = "one_percent"
path = base_path / folname
up_graph_set = graphs_001

In [ ]:
q_func = "update_queue_remove_asymmetric"
folname = "one_percent"
path = base_path / folname
up_graph_set = graphs_001